# 2 · Oscillators Hold a Phase in Time

*Phasor networks, from the ground up — notebook 2 of 7.*

A phase becomes computational when an **oscillator** carries it. A resonate-and-fire
neuron has a complex membrane state `z` obeying

$$\frac{dz}{dt} = k\,z + I(t), \qquad k = \lambda + i\omega.$$

Left alone, `z` rotates at the carrier frequency `ω` and decays at rate `λ`. Driven
by an input spike, it locks to a **stable phase** and holds it cycle after cycle.
`oscillator_bank` integrates a whole bank of these neurons. We drive a bank with
encoded phases, watch them lock in, and read the phases back — the round trip
*spike → oscillator → spike → phase* that underlies everything later.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots
using DifferentialEquations

## Encode a sweep of phases as spikes

One neuron per phase, swept over `[-1, 1]`, encoded over several cycles.

In [ ]:
n_x = 101
phases = reshape(collect(range(-1.0, 1.0, n_x)), (:, 1))

spk_args = SpikingArgs()
repeats = 6
tspan = (0.0, repeats * 1.0)
tbase = collect(tspan[1]:spk_args.solver_args[:dt]:tspan[2])

st = phase_to_train(phases, spk_args=spk_args, repeats=repeats, offset=0.0f0);

## Integrate the bank

`oscillator_bank` returns the ODE solution: the complex state of every neuron at every timestep.

In [ ]:
sol = oscillator_bank(st, tspan=tspan, spk_args=spk_args)
u = Array(sol);
size(u)

A single neuron's state rotates and settles onto a circle — it is *holding* a phase: a steady angle that repeats every period.

In [ ]:
plot(real.(u[1, 1, :]), label="real", xlabel="timestep", ylabel="state")
plot!(imag.(u[1, 1, :]), label="imag", title="one neuron's state over time")

In [ ]:
scatter(real.(u[1, 1, :]), imag.(u[1, 1, :]), label="", aspect_ratio=:equal,
        xlabel="real", ylabel="imag", title="the state holds a phase (a fixed angle)", markersize=2)

## Read the held phase back

`solution_to_phase` extracts the carried phase at each cycle. Neuron 51 (phase ≈ 0) locks to its value.

In [ ]:
p = solution_to_phase(sol, spk_args=spk_args, final_t=false)
plot(p[51, 1, :], xlabel="cycle", ylabel="phase", label="",
     title="neuron 51 locks to its phase")

## Close the loop

Convert the settled solution back to spikes (`solution_to_train`) and decode (`train_to_phase`). The recovered phases match the inputs to within the spike-time resolution — the oscillator faithfully held every phase.

In [ ]:
st1 = solution_to_train(sol, tspan, spk_args=spk_args, offset=0.0)
p1 = train_to_phase(st1, spk_args=spk_args)

recovered = p1[end-1, :, 1]
err = arc_error(recovered .- vec(phases))
println("max arc error: ", maximum(err))

scatter(recovered, vec(phases), xlabel="recovered phase", ylabel="input phase",
        label="", title="spike → oscillator → spike → phase")
plot!(-1:1, -1:1, label="y = x", linestyle=:dash)

## Next

An oscillator carries one phase. Two or more oscillators, *connected*, compute the
hyperdimensional operations — similarity, bundling, binding — by superposition and
rotation of their states. That is notebook 3.